# Modelo elegido y explicación con SHAP – Riesgo de prórroga en contratos SECOP II

**Modelo:** LightGBM (`lgbm-base`), registrado en MLflow como `secop-prorroga` v1 y exportado en `models/modelo_prorroga.joblib`.

Este notebook revisa el modelo con los datos de **validación** (contratos firmados entre abril y junio de 2024):
1. qué tan bien ordena y si sus probabilidades son realistas (calibración);
2. **qué variables usa para predecir la prórroga** (SHAP).

Requiere `dvc pull` (datos y `models/`).

In [ ]:
import sys
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score

ROOT = Path("..") if Path.cwd().name == "notebooks" else Path(".")
sys.path.insert(0, str(ROOT / "src"))
from features import CATEGORICAS, NUMERICAS, OBJETIVO

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 60, "display.float_format", "{:,.3f}".format)

modelo = joblib.load(ROOT / "models/modelo_prorroga.joblib")
df = pd.read_parquet(ROOT / "data/processed/features.parquet")
VARIABLES = CATEGORICAS + NUMERICAS
print(f"{len(df):,} contratos · {len(CATEGORICAS)} categóricas + {len(NUMERICAS)} numéricas")
df["split"].value_counts()

## 1. Desempeño en validación

El modelo da una probabilidad de prórroga para cada contrato. El **AUC** mide qué tan bien ordena: la probabilidad de que un contrato prorrogado quede por encima de uno que no.
La métrica de negocio es la **precisión en el 10 % de mayor riesgo**: de los contratos que el tablero pondría en alerta, cuántos se prorrogaron.

In [ ]:
valid = df[df["split"] == "valid"].copy()
valid["prob"] = modelo.predict_proba(valid[VARIABLES])[:, 1]

top10 = valid.nlargest(int(len(valid) * 0.10), "prob")
print(f"AUC: {roc_auc_score(valid[OBJETIVO], valid['prob']):.3f}")
print(f"Tasa de prórroga en validación: {valid[OBJETIVO].mean():.1%}")
print(f"Precisión en el 10 % de mayor riesgo: {top10[OBJETIVO].mean():.1%}")

## 2. Calibración

Agrupamos los contratos según la probabilidad que dio el modelo y comparamos con la fracción que **de verdad** se prorrogó. Si las dos columnas se parecen, el modelo está calibrado: cuando dice 40 %, se prorroga cerca del 40 %.

In [ ]:
cortes = [0, .05, .10, .15, .25, .35, .50, .70, 1]
calib = (valid.assign(tramo=pd.cut(valid["prob"], cortes))
              .groupby("tramo", observed=True)
              .agg(contratos=(OBJETIVO, "size"), dijo_el_modelo=("prob", "mean"), paso_de_verdad=(OBJETIVO, "mean")))
calib

## 3. Valores SHAP: qué hay detrás de cada predicción

Para **cada contrato**, SHAP reparte la predicción entre las 40 variables: cuánto subió (+) o bajó (−) el riesgo cada una. LightGBM los calcula directamente con `pred_contrib=True`.

- Resultado: una matriz de **contratos × variables**, más una columna `base` (el punto de partida, igual para todos).
- Están en **log-odds**, no en porcentaje: sirven para comparar variables y ver la dirección del efecto.
- `base` + la suma de una fila = la predicción de ese contrato (en log-odds). Lo comprobamos abajo.

Usamos una muestra de 20.000 contratos para que sea rápido.

In [ ]:
muestra = valid.sample(20_000, random_state=42)
X = muestra[VARIABLES]

prep, lgbm = modelo[:-1], modelo[-1]      # el pipeline: preparación + LightGBM
contrib = lgbm.predict(prep.transform(X), pred_contrib=True)

shap_df = pd.DataFrame(contrib[:, :-1], columns=VARIABLES, index=muestra.index)
base = contrib[0, -1]
print("Matriz SHAP:", shap_df.shape, "· base =", round(base, 3))
shap_df.head()

In [ ]:
# Comprobación: base + suma de los SHAP = predicción del modelo
log_odds = base + shap_df.sum(axis=1)
prob_desde_shap = 1 / (1 + np.exp(-log_odds))
print("Diferencia máxima con predict_proba:", np.abs(prob_desde_shap - muestra["prob"]).max())

## 4. Importancia global (todas las variables)

La importancia de una variable es el **promedio del tamaño** de su SHAP (valor absoluto), sin importar si sube o baja el riesgo. Aplica igual a continuas y categóricas.

In [ ]:
importancia = (shap_df.abs().mean()
               .sort_values(ascending=False)
               .to_frame("shap_medio_abs"))
importancia["porcentaje"] = importancia["shap_medio_abs"] / importancia["shap_medio_abs"].sum()
importancia["tipo"] = np.where(importancia.index.isin(CATEGORICAS), "categórica", "numérica")
importancia.head(15)

In [ ]:
top = importancia.head(15).iloc[::-1]
colores = top["tipo"].map({"numérica": "#1F5FAF", "categórica": "#C2410C"})
plt.figure(figsize=(8, 6))
plt.barh(top.index, top["shap_medio_abs"], color=colores)
plt.xlabel("Promedio de |SHAP|")
plt.title("Variables que más pesan (azul: numérica, naranja: categórica)")
plt.show()

## 5. Dirección del efecto en variables **continuas**

Para una variable continua miramos si un valor **alto** sube o baja el riesgo. Cada punto es un contrato: en el eje X el valor de la variable, en el eje Y su SHAP (arriba de 0 sube el riesgo).

Cambia `variable` por otra de `NUMERICAS` para explorarla.

In [ ]:
NUMERICAS

In [ ]:
variable = "duracion_pactada_dias"

plt.figure(figsize=(8, 4))
plt.scatter(X[variable], shap_df[variable], s=4, alpha=0.3)
plt.axhline(0, color="gray", lw=1)
plt.xlabel(variable)
plt.ylabel("SHAP (efecto en el riesgo)")
plt.title(f"Efecto de {variable}")
plt.show()

In [ ]:
# Vista resumida de todas las numéricas (rojo = valor alto de la variable)
import shap
orden = [c for c in importancia.index if c in NUMERICAS][:12]
shap.summary_plot(shap_df[orden].to_numpy(), X[orden].astype(float), feature_names=orden, max_display=12)

## 6. Dirección del efecto en variables **categóricas**

Una categórica no tiene valores "altos" o "bajos", así que promediamos el SHAP de **cada categoría**: positivo sube el riesgo, negativo lo baja. Se muestran solo las categorías con al menos 100 contratos en la muestra.

Cambia `variable` por otra de `CATEGORICAS`.

In [ ]:
CATEGORICAS

In [ ]:
def efecto_por_categoria(variable, min_n=100):
    return (pd.DataFrame({"categoria": X[variable], "shap": shap_df[variable]})
              .groupby("categoria")["shap"].agg(contratos="size", shap_medio="mean")
              .query("contratos >= @min_n")
              .sort_values("shap_medio", ascending=False))

efecto_por_categoria("departamento")

In [ ]:
efecto_por_categoria("sector")

## 7. Un contrato en particular: ¿por qué el modelo le dio ese riesgo?

Tomamos el contrato con mayor probabilidad de la muestra y ordenamos sus SHAP. Esto es lo que el tablero muestra como "factores que más aumentan el riesgo".

Cambia `id_contrato` por el índice de otro contrato (por ejemplo, uno de `muestra.sample(1).index`).

In [ ]:
id_contrato = muestra["prob"].idxmax()
c = muestra.loc[id_contrato]
print(f"{c['tipo_de_contrato']} · {c['departamento']} · {c['duracion_pactada_dias']:.0f} días · "
      f"probabilidad {c['prob']:.1%} · ¿se prorrogó? {'sí' if c[OBJETIVO] else 'no'}")

factores = (pd.DataFrame({"valor": X.loc[id_contrato], "shap": shap_df.loc[id_contrato]})
              .sort_values("shap", key=abs, ascending=False).head(10))
factores

In [ ]:
f = factores.iloc[::-1]
plt.figure(figsize=(8, 4))
plt.barh(f.index, f["shap"], color=np.where(f["shap"] > 0, "#C2410C", "#1F5FAF"))
plt.axvline(0, color="gray", lw=1)
plt.xlabel("SHAP (naranja sube el riesgo, azul lo baja)")
plt.title("Las 10 variables que más movieron la predicción de este contrato")
plt.show()

## 8. Para tener en cuenta

- SHAP describe **qué usa el modelo para predecir**, no qué **causa** la prórroga.
- Cada efecto se lee "manteniendo las demás variables": por ejemplo, el efecto del precio base es el que queda después de considerar la duración.
- Preguntas para seguir explorando: ¿el efecto del precio cambia según el tipo de contrato? ¿Qué departamentos suben el riesgo aunque su tasa histórica sea baja?